# Supplementary Results 3 — Systematic gene prioritisation using the L2G model

The feature matrix, the training and held-out sets, the model's performance on the held-out set,
and what supports each prioritised gene.

Numbers are written to `results/sr03_l2g.json`.

**Provenance.** The feature-matrix and prioritisation counts come from
`~/Projects/EGL_and_training_set/archive/gentropy_paper/02_descriptive_numbers_si_vi_fm.ipynb`
(cells 124-182) and `14_gene_stauration_plots.ipynb`. The training and held-out sets are
`data/l2g_training_set/`, recovered from the same archive; see GAPS.md section 1. The novelty
comparison against Open Targets Genetics 22.10 has no input in this repository and is precomputed.

**Four published numbers in this section disagree with the code that produced their neighbours.**
Each is flagged where it arises, with the archive cell that settles it:

| Published | Here | Settled by |
| --- | --- | --- |
| 17,463 CSs with more than one gene at L2G >= 0.5 (3.4%), 193,523 with none (37.1%) | 14,196 (2.7%) and 196,790 (37.8%) | the archive cell that prints 309,989 prints 13,474 twice and 722 more than twice in the same breath: 13,474 + 722 = 14,196 |
| true negatives 17,362 in the Supplementary Figure SR1 caption | 17,382 | 17,362 + 95 false positives is 17,457, twenty short of the section's own 17,477 held-out negatives; 17,382 + 95 is exact |
| 13.0% of prioritisations supported by a PAV | 12.1% | the same sentence's count, 63,327, is 12.1% of 523,409 |
| 18,950 genes (94.1%) once Orphanet, gene burden and eQTLs are added | 18,813 (93.5%) | the archive's saturation notebook reports 18,809 of 20,083 protein-coding genes, 93.7% |

In [1]:
import numpy as np
import pandas as pd
from gentropy.common.session import Session
from pyspark.sql import functions as f
from sklearn import metrics

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
numbers = {}

# The manuscript's own L2G model, not the release's — the same predictions
# `01-data-preparation/05_l2g_prioritised_genes.ipynb` builds every prioritisation from.
L2G_PREDICTIONS = str(paper.ROOT / "data" / "25.06" / "irene_1208_l2g_predictions")
TRAINING_SET = paper.ROOT / "data" / "l2g_training_set" / "20250625_gentropy_paper_v1"
TEST_SET = paper.ROOT / "data" / "l2g_training_set" / "test_v3.parquet"

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/20 15:08:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Precomputed numbers

The novelty comparison is **precomputed**: it was run against Open Targets Genetics release 22.10,
which this repository does not hold and which predates the pipeline the rest of the section is built
on. Nothing below reproduces these three values, and they are reported as PRECOMPUTED rather than as
failures.

In [2]:
PRECOMPUTED = pd.DataFrame(
    [
        {
            "id": "S3.42",
            "claim": "GWAS credible sets considered novel against Open Targets Genetics 22.10",
            "value": 456323,
            "why it is not recomputed": "the reference is OTG release 22.10, a data set this "
            "repository does not download; nothing in 25.06 identifies which credible sets were "
            "already known there",
            "input that would close it": "the OTG 22.10 study-locus release",
        },
        {
            "id": "S3.43",
            "claim": "those as a share of all GWAS credible sets (%)",
            "value": 58.0,
            "why it is not recomputed": "as above",
            "input that would close it": "as above",
        },
        {
            "id": "S3.44",
            "claim": "GWAS credible sets classified as previously known",
            "value": 333130,
            "why it is not recomputed": "as above",
            "input that would close it": "as above",
        },
    ]
)
PRECOMPUTED

,id,claim,value,why it is not recomputed,input that would close it
0,S3.42,GWAS credible sets considered novel against Op...,456323.0,"the reference is OTG release 22.10, a data set...",the OTG 22.10 study-locus release
1,S3.43,those as a share of all GWAS credible sets (%),58.0,as above,as above
2,S3.44,GWAS credible sets classified as previously known,333130.0,as above,as above


## The feature matrix

Restricted to protein-coding genes and to qualifying credible sets, which is the matrix the model
was applied to.

In [3]:
qualifying_cs = (
    session.spark.read.parquet(paper.derived("qualifying_credible_sets"))
    .select("studyLocusId")
    .union(session.spark.read.parquet(paper.derived("qualifying_measurement_credible_sets")).select("studyLocusId"))
    .distinct()
    .cache()
)
n_qualifying = qualifying_cs.count()

fm = (
    session.spark.read.parquet(paper.release("l2g_feature_matrix"))
    .filter(f.col("isProteinCoding") == 1)
    .select("studyLocusId", "geneId")
    .join(qualifying_cs, "studyLocusId", "inner")
    .cache()
)
numbers["S3.02"] = fm.count()
numbers["S3.01"] = fm.select("studyLocusId").distinct().count()

per_cs = fm.groupBy("studyLocusId").agg(f.countDistinct("geneId").alias("genes"))
spread = per_cs.agg(f.mean("genes").alias("mean"), f.expr("percentile_approx(genes, 0.5)").alias("median")).collect()[0]
numbers["S3.03"] = round(float(spread["mean"]), 2)
numbers["S3.04"] = int(spread["median"])
print({k: numbers[k] for k in ["S3.01", "S3.02", "S3.03", "S3.04"]})

{'S3.01': 513568, 'S3.02': 7066749, 'S3.03': 13.76, 'S3.04': 10}


## The training and held-out sets

`20250625_gentropy_paper_v1` is the full gold standard; `test_v3.parquet` is the saved held-out
split. The split seed was never recorded, so the saved file is the only way to reproduce it, and
the training set is the gold standard minus those rows.

In [4]:
gold = session.spark.read.parquet(str(TRAINING_SET)).cache()
test = pd.read_parquet(TEST_SET)
print(f"gold standard rows: {gold.count():,} | held-out rows: {len(test):,}")

positives = gold.filter(f.col("goldStandardSet") == "positive").cache()
numbers["S3.05"] = positives.count()
numbers["S3.07"] = positives.select("geneId").distinct().count()

# The published 1,377 counts positives that survive into the protein-coding feature matrix, and
# counts distinct `diseaseIds` **arrays** rather than exploded terms — from
# `fm_pos.dropDuplicates(["geneId", "diseaseIds"])` in the archive's
# 02_descriptive_numbers_si_vi_fm.ipynb, cells 163-167. Exploding instead gives 1,704, the number
# Supplementary Table 13 reports.
matrix_positives = (
    session.spark.read.parquet(paper.release("l2g_feature_matrix"))
    .filter(f.col("isProteinCoding") == 1)
    .select("studyLocusId", "geneId", "distanceSentinelTssNeighbourhood")
    .join(positives.select("studyLocusId", "geneId", "diseaseIds"), ["studyLocusId", "geneId"], "inner")
    .cache()
)
numbers["S3.06"] = matrix_positives.select("geneId", "diseaseIds").distinct().count()
exploded = matrix_positives.select("geneId", f.explode("diseaseIds").alias("diseaseId")).distinct().count()
print(f"positive gene-EFO pairs: {numbers['S3.06']:,} as trait combinations, {exploded:,} exploded")

numbers["S3.10"] = int((test["goldStandardSet"] == 1).sum())
numbers["S3.11"] = int((test["goldStandardSet"] == 0).sum())
numbers["S3.08"] = numbers["S3.05"] - numbers["S3.10"]
numbers["S3.09"] = gold.filter(f.col("goldStandardSet") == "negative").count() - numbers["S3.11"]
numbers["S3.12"] = round((numbers["S3.09"] + numbers["S3.11"]) / (numbers["S3.08"] + numbers["S3.10"]), 1)
print({k: numbers[k] for k in ["S3.05", "S3.06", "S3.07", "S3.08", "S3.09", "S3.10", "S3.11", "S3.12"]})

gold standard rows: 132,970 | held-out rows: 18,611


positive gene-EFO pairs: 1,377 as trait combinations, 1,704 exploded
{'S3.05': 8520, 'S3.06': 1377, 'S3.07': 390, 'S3.08': 7386, 'S3.09': 106973, 'S3.10': 1134, 'S3.11': 17477, 'S3.12': 14.6}


In [5]:
# How often the positive gene is the one whose TSS is nearest the lead variant.
nearest = (
    matrix_positives.filter(f.col("distanceSentinelTssNeighbourhood") == 1)
    .select("geneId", "diseaseIds")
    .distinct()
    .count()
)
numbers["S3.13"] = nearest
numbers["S3.14"] = round(100 * nearest / numbers["S3.06"], 1)
print(f"positive gene-EFO pairs whose gene is nearest to the TSS: {nearest:,} ({numbers['S3.14']}%)")

positive gene-EFO pairs whose gene is nearest to the TSS: 773 (56.1%)


## Model performance on the held-out set

The trained model itself was never saved, but its predictions were: `irene_1208_l2g_predictions`
is the manuscript's model applied to every credible set, and it is what every prioritisation in
this pipeline is built from. That table is **floored at 0.05**, so it can reproduce anything that
depends only on the 0.5 threshold — precision, recall, the confusion matrix — but not the two
metrics that depend on the ranking of the low-scoring rows. Coverage is printed first, then the
metrics, then what the floor costs.

In [6]:
predictions = session.spark.read.parquet(L2G_PREDICTIONS).select("studyLocusId", "geneId", "score")
scored = (
    session.spark.createDataFrame(test[["studyLocusId", "geneId", "goldStandardSet"]])
    .join(predictions, ["studyLocusId", "geneId"], "left")
    .toPandas()
)
covered = scored["score"].notna()
print(f"held-out rows with a released L2G score: {covered.sum():,} of {len(scored):,} ({100 * covered.mean():.1f}%)")

# That coverage is not a vintage mismatch: every held-out credible set and every held-out CS-gene
# pair is present in the 25.06 feature matrix. The prediction table is **floored at 0.05** — the
# minimum score it holds — so the pairs with no row are pairs the model scored below 0.05, and
# filling them with 0 is right in kind. What it destroys is their order: they all become one tie.
print(f"lowest score in the prediction table: {predictions.agg(f.min('score')).collect()[0][0]}")
scored["score"] = scored["score"].fillna(0.0)

held-out rows with a released L2G score: 2,148 of 18,611 (11.5%)
lowest score in the prediction table: 0.050000082701444626


In [7]:
y = scored["goldStandardSet"].astype(int)
p = scored["score"]
predicted = (p >= 0.5).astype(int)
tn, fp, fn, tp = metrics.confusion_matrix(y, predicted).ravel()

# AP is the area under the precision-recall curve. sklearn's `average_precision_score` is the
# step-wise summary of the same curve, which is lower here because of the tie at 0; both are
# printed, and the trapezoidal area is the one registered.
precision, recall, _ = metrics.precision_recall_curve(y, p)
order = recall.argsort()
numbers["S3.15"] = round(float(metrics.auc(recall[order], precision[order])), 2)
numbers["S3.16"] = round(float(metrics.roc_auc_score(y, p)), 2)
numbers["S3.17"] = round(float(tp / (tp + fp)), 3)
numbers["S3.18"] = round(float(tn / (tn + fp)), 3)
numbers["S3.19"] = round(float(tp / (tp + fn)), 3)
numbers["S3.20"], numbers["S3.21"], numbers["S3.22"], numbers["S3.23"] = int(tn), int(fp), int(fn), int(tp)
print(
    f"area under the PR curve {numbers['S3.15']} "
    f"(step-wise average precision {metrics.average_precision_score(y, p):.3f}), "
    f"ROC AUC {numbers['S3.16']}"
)
print(f"at 0.5: precision {numbers['S3.17']}, selectivity {numbers['S3.18']}, recall {numbers['S3.19']}")
print(f"confusion matrix: TN {tn:,} FP {fp:,} FN {fn:,} TP {tp:,}")
# The published caption reports TN 17,362, which with its own 95 false positives makes 17,457 —
# twenty short of the 17,477 held-out negatives the same section reports. TN 17,382 + FP 95 is exact.
print(f"TN + FP = {tn + fp:,}, against {numbers['S3.11']:,} held-out negatives")

area under the PR curve 0.78 (step-wise average precision 0.756), ROC AUC 0.91
at 0.5: precision 0.885, selectivity 0.995, recall 0.646
confusion matrix: TN 17,382 FP 95 FN 402 TP 732
TN + FP = 17,477, against 17,477 held-out negatives


### What the 0.05 floor costs the two ranking metrics

Precision, recall and the confusion matrix only need the threshold at 0.5, so they are unaffected
and they reproduce. Average precision and the ROC AUC need the *order* of the low-scoring rows, and
that order is exactly what the floor removes. The cell below brackets the ROC AUC: every
positive-negative pair inside the tied band currently earns half credit, so the true value lies
between giving that band no credit and giving it full credit. The published 0.95 sits inside the
bracket, and the published 0.81 is likewise above the trapezoidal 0.78 computed under the tie.

In [8]:
band = scored["score"] == 0
n_pos, n_neg = int(y.sum()), int((1 - y).sum())
pos_band, neg_band = int((band & (y == 1)).sum()), int((band & (y == 0)).sum())
tied_share = pos_band * neg_band / (n_pos * n_neg)
print(f"tied at 0: {pos_band:,} of {n_pos:,} positives and {neg_band:,} of {n_neg:,} negatives")
print(f"they make up {100 * tied_share:.1f}% of all positive-negative pairs, each scored at half credit")
print(
    f"ROC AUC bracket: {numbers['S3.16'] - tied_share / 2:.3f} to {numbers['S3.16'] + tied_share / 2:.3f} "
    f"(published 0.95)"
)

tied at 0: 183 of 1,134 positives and 16,280 of 17,477 negatives
they make up 15.0% of all positive-negative pairs, each scored at half credit
ROC AUC bracket: 0.835 to 0.985 (published 0.95)


## How many genes the model prioritises per credible set

In [9]:
scores_on_qualifying = (
    session.spark.read.parquet(L2G_PREDICTIONS)
    .select("studyLocusId", "geneId", "score")
    .join(qualifying_cs, "studyLocusId", "inner")
    .cache()
)


def genes_per_cs(scores, label):
    """How many qualifying credible sets carry none, one or several genes at L2G >= 0.5."""
    above = scores.filter(f.col("score") >= 0.5).groupBy("studyLocusId").agg(f.countDistinct("geneId").alias("genes"))
    one = above.filter(f.col("genes") == 1).count()
    many = above.filter(f.col("genes") > 1).count()
    none = n_qualifying - one - many
    return {
        "gene set": label,
        "exactly one": one,
        "% one": round(100 * one / n_qualifying, 1),
        "more than one": many,
        "% more": round(100 * many / n_qualifying, 1),
        "none": none,
        "% none": round(100 * none / n_qualifying, 1),
    }


per_cs_summary = pd.DataFrame(
    [
        genes_per_cs(scores_on_qualifying, "every scored gene"),
        genes_per_cs(
            scores_on_qualifying.join(fm.select("studyLocusId", "geneId"), ["studyLocusId", "geneId"]),
            "protein-coding genes only",
        ),
    ]
)
# The archive cell that produces the published 309,989 produces this split too:
# `02_descriptive_numbers_si_vi_fm.ipynb` cell 182 prints "met only once: 309989", "met twice:
# 13474" and "met more than twice: 722" — so more than one gene is 13,474 + 722 = 14,196, and the
# published 17,463 appears in no surviving notebook. 193,523 is its complement and moves with it.
row = per_cs_summary.iloc[1]
numbers["S3.24"], numbers["S3.25"] = int(row["exactly one"]), float(row["% one"])
numbers["S3.26"], numbers["S3.27"] = int(row["more than one"]), float(row["% more"])
numbers["S3.28"], numbers["S3.29"] = int(row["none"]), float(row["% none"])
per_cs_summary

,gene set,exactly one,% one,more than one,% more,none,% none
0,every scored gene,309989,59.5,14196,2.7,196790,37.8
1,protein-coding genes only,309989,59.5,14196,2.7,196790,37.8


## What supports each prioritised gene

The published counts settle a denominator the main text leaves ambiguous: 241,404 / 523,409 =
46.1%, so "46.1% having no PAV, eQTL or pQTL evidence" is over **all** prioritised genes, not over
the 81.2% that are nearest to a TSS. Both are computed here.

In [10]:
assignments = (
    session.spark.read.parquet(paper.derived("prioritised_genes_diseases"))
    .select("studyLocusId", "geneId", "eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS")
    .unionByName(
        session.spark.read.parquet(paper.derived("prioritised_genes_measurements")).select(
            "studyLocusId", "geneId", "eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS"
        )
    )
    .distinct()
    .toPandas()
)
total = len(assignments)
unsupported = (assignments["VEP"] == 0) & (assignments["eQTL_coloc"] == 0) & (assignments["pQTL_coloc"] == 0)

support = pd.DataFrame(
    [
        {"support": "eQTL colocalisation", "assignments": int(assignments["eQTL_coloc"].sum())},
        {"support": "pQTL colocalisation", "assignments": int(assignments["pQTL_coloc"].sum())},
        {"support": "protein-altering variant", "assignments": int(assignments["VEP"].sum())},
        {"support": "nearest to a TSS", "assignments": int(assignments["distanceTSS"].sum())},
        {"support": "none of PAV, eQTL, pQTL", "assignments": int(unsupported.sum())},
        {
            "support": "nearest to a TSS and none of the three",
            "assignments": int(((assignments["distanceTSS"] == 1) & unsupported).sum()),
        },
    ]
)
support["% of all assignments"] = (100 * support["assignments"] / total).round(1)
print(f"CS-gene prioritisations: {total:,}")

numbers["S3.30"], numbers["S3.31"] = support.loc[0, "assignments"], float(support.loc[0, "% of all assignments"])
numbers["S3.32"], numbers["S3.33"] = support.loc[1, "assignments"], float(support.loc[1, "% of all assignments"])
numbers["S3.34"], numbers["S3.35"] = support.loc[2, "assignments"], float(support.loc[2, "% of all assignments"])
numbers["S3.36"], numbers["S3.37"] = support.loc[3, "assignments"], float(support.loc[3, "% of all assignments"])
# The published pair (241,404 = 46.1%) is the nearest-gene assignments that carry none of the three,
# as a share of **all** assignments. Over the nearest-gene assignments alone the same count reads
# 56.8%, which is what `02-analysis-main/03_colocalisation_l2g.ipynb` reported for R3.16.
numbers["S3.38"], numbers["S3.39"] = support.loc[5, "assignments"], float(support.loc[5, "% of all assignments"])

nearest_only = assignments[assignments["distanceTSS"] == 1]
nearest_unsupported = (
    (nearest_only["VEP"] == 0) & (nearest_only["eQTL_coloc"] == 0) & (nearest_only["pQTL_coloc"] == 0)
).mean()
print(f"among nearest-gene assignments alone, unsupported: {100 * nearest_unsupported:.1f}%")
support

CS-gene prioritisations: 523,409
among nearest-gene assignments alone, unsupported: 56.8%


,support,assignments,% of all assignments
0,eQTL colocalisation,191871,36.7
1,pQTL colocalisation,30030,5.7
2,protein-altering variant,63327,12.1
3,nearest to a TSS,424781,81.2
4,"none of PAV, eQTL, pQTL",282039,53.9
5,nearest to a TSS and none of the three,241404,46.1


## Genes associated with any human trait when other resources are added

The claim combines the prioritised genes with Orphanet, gene-burden associations and eQTLs, and
reports the union as a share of all protein-coding genes. Which eQTL set is meant is not stated,
so both readings are computed: every gene with a molQTL credible set, and only those whose molQTL
credible set colocalises with a qualifying credible set.

In [11]:
evidence = session.spark.read.parquet(paper.release("evidence")).select("targetId", "datasourceId").cache()
print(evidence.groupBy("datasourceId").count().orderBy(f.desc("count")).toPandas().head(20).to_string(index=False))

      datasourceId    count
         europepmc 22168394
               eva  3553201
gwas_credible_sets  1560570
              impc  1188863
            chembl   573103
  expression_atlas   229398
cancer_gene_census    82754
        slapenrich    72405
       gene_burden    38223
  genomics_england    34841
  uniprot_variants    33047
     crispr_screen    21700
          reactome    10166
       eva_somatic     9830
uniprot_literature     6744
          orphanet     6301
           intogen     4224
    gene2phenotype     3304
           clingen     3100
 cancer_biomarkers     1300


In [12]:
target = session.spark.read.parquet(paper.release("target")).select(f.col("id").alias("geneId"), "biotype")
protein_coding = target.filter(f.col("biotype") == "protein_coding").select("geneId").cache()
n_protein_coding = protein_coding.count()

prioritised = (
    session.spark.read.parquet(paper.derived("prioritised_genes_diseases"))
    .select("geneId")
    .union(session.spark.read.parquet(paper.derived("prioritised_genes_measurements")).select("geneId"))
    .distinct()
)
orphanet = evidence.filter(f.col("datasourceId") == "orphanet").select(f.col("targetId").alias("geneId")).distinct()
burden = evidence.filter(f.col("datasourceId") == "gene_burden").select(f.col("targetId").alias("geneId")).distinct()
molqtl_genes = (
    session.spark.read.parquet(paper.release("credible_set"))
    .filter(f.col("studyType") != "gwas")
    .select("studyId")
    .distinct()
    .join(session.spark.read.parquet(paper.release("study")).select("studyId", "geneId"), "studyId", "inner")
    .select("geneId")
    .distinct()
)


def union_size(parts, label):
    """Protein-coding genes in the union of several gene sets."""
    combined = parts[0]
    for part in parts[1:]:
        combined = combined.union(part)
    n = combined.distinct().join(protein_coding, "geneId").count()
    return {"union": label, "genes": n, "% of protein-coding": round(100 * n / n_protein_coding, 1)}


clingen = evidence.filter(f.col("datasourceId") == "clingen").select(f.col("targetId").alias("geneId")).distinct()
unions = pd.DataFrame(
    [
        union_size([prioritised, orphanet, burden, molqtl_genes], "prioritised + Orphanet + burden + molQTL"),
        union_size([prioritised, orphanet, burden], "prioritised + Orphanet + burden"),
        union_size([prioritised, orphanet, burden, molqtl_genes, clingen], "and ClinGen as well"),
    ]
)
# The archive's saturation notebook (`14_gene_stauration_plots.ipynb`, cells 83-88) unions
# Orphanet, OMIM, gene burden, ChEMBL, the L2G disease and measurement predictions, the L2G eQTL
# and VEP evidence and the molQTL genes, and reports 18,809 of 20,083 protein-coding genes on
# valid chromosomes — 93.7%, within four genes of the union computed here. The published 18,950
# (94.1%) is in neither.
numbers["S3.40"] = int(unions.loc[0, "genes"])
numbers["S3.41"] = float(unions.loc[0, "% of protein-coding"])
print(f"protein-coding genes in the release: {n_protein_coding:,}")
unions

protein-coding genes in the release: 20,130


,union,genes,% of protein-coding
0,prioritised + Orphanet + burden + molQTL,18813,93.5
1,prioritised + Orphanet + burden,16335,81.1
2,and ClinGen as well,18817,93.5


## Write the results

In [13]:
print(paper.save_results("sr03_l2g", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr03_l2g.json


,computed
S3.02,7066749.000
S3.01,513568.000
S3.03,13.760
S3.04,10.000
S3.05,8520.000
S3.07,390.000
S3.06,1377.000
S3.10,1134.000
S3.11,17477.000
S3.08,7386.000
